In [2]:
# exercise_data_io

from pyspark import SparkContext

sc = SparkContext("local[*]", "DataIO")


# load the CSV file
lines = sc.textFile("sales_data.csv")

# skip the header line
header = lines.first()
data = lines.filter(lambda line: line != header)
print(f"header: {header}")
print(f"Data records: {data.count()}")
print(f"First record: {data.first()}")


def parse_record(line):
    """Parse CSV line into structured data."""
    parts = line.split(",")
    return {
        "product_id": parts[0],
        "name": parts[1],
        "category": parts[2],
        "price": float(parts[3]),
        "quantity": int(parts[4])
    }

# Parse all records
parsed = data.map(parse_record)

# Show parsed data
for record in parsed.take(3):
    print(record)


# Calculate revenue for each product
revenue = parsed.map(lambda r: f"{r['product_id']},{r['name']},{r['price'] * r['quantity']:.2f}")

# Save to output directory
# YOUR CODE: Use saveAsTextFile to save revenue data
# revenue.saveAsTextFile("final")


# reading both files in using the wildcard
all_data = sc.textFile("sales_data*.csv")
header = all_data.first()
filtered = all_data.filter(lambda line: line != header)

parsed2 = filtered.map(parse_record)

li = parsed2.collect()

print("this is the data for all files")
for record in li:
    print(record)

revenue = parsed.map(lambda r: f"{r['product_id']},{r['name']},{r['price'] * r['quantity']:.2f}")

merge = revenue.coalesce(1)
merge.saveAsTextFile("merged")



ValueError: Cannot run multiple SparkContexts at once; existing SparkContext(app=DataIO, master=local[*]) created by __init__ at /tmp/ipykernel_119342/4089981496.py:5 

In [1]:
from pyspark import SparkContext

sc = SparkContext("local[*]", "RDDActions")

numbers = sc.parallelize([10, 5, 8, 3, 15, 12, 7, 20, 1, 9])

# Task A: collect() - Get all elements
all_nums = numbers.collect()
print(f"All numbers: {all_nums}")

# Task B: count() - Count elements
count = numbers.count()
print(f"Count: {count}")

# Task C: first() - Get first element
first = numbers.first()
print(f"First: {first}")

# Task D: take(n) - Get first n elements
first_three = numbers.take(4)
print(f"First 3: {first_three}")

# Task E: top(n) - Get largest n elements
top_three = numbers.top(3)
print(f"Top 3: {top_three}")

# Task F: takeOrdered(n) - Get smallest n elements
smallest_three = numbers.takeOrdered(3)
print(f"Smallest 3: {smallest_three}")

#----------------- Part 2 ---------------

# Task A: reduce() - Sum all numbers
total = numbers.reduce(lambda x, y: x + y) # YOUR CODE using reduce with lambda
print(f"Sum: {total}")

# Task B: reduce() - Find maximum
maximum = numbers.reduce(max)
print(f"Max: {maximum}")

# Task C: reduce() - Find minimum
minimum = numbers.reduce(min) # YOUR CODE using reduce
print(f"Min: {minimum}")

# Task D: fold() - Sum with zero value
folded_sum = numbers.fold(0, lambda x,y: x+y)# YOUR CODE
print(f"Folded sum: {folded_sum}")


#----------------- Part 3 ---------------
# Given: colors with duplicates
colors = sc.parallelize(["red", "blue", "red", "green", "blue", "red", "yellow"])

# Count occurrences of each color
color_counts = (colors.map(lambda color: (color, 1)).reduceByKey(lambda x, y: x + y).collect()) # YOUR CODE
print(f"Color counts: {dict(color_counts)}")
sc.stop()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/29 20:45:39 WARN Utils: Your hostname, DaniyalsLenovo, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/06/29 20:45:39 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/29 20:45:40 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


All numbers: [10, 5, 8, 3, 15, 12, 7, 20, 1, 9]


Count: 10
First: 10
First 3: [10, 5, 8, 3]
Top 3: [20, 15, 12]
Smallest 3: [1, 3, 5]
Sum: 90
Max: 20
Min: 1
Folded sum: 90


Color counts: {'blue': 2, 'green': 1, 'yellow': 1, 'red': 3}


In [2]:
#RDD basics

from pyspark import SparkContext


sc = SparkContext("local[*]", "RDDBasics")

# 1. Create RDD from a Python list
numbers = sc.parallelize([1, 2, 3, 4, 5, 6, 7, 8, 9, 10])
print(f"Numbers: {numbers.collect()}")
print(f"Partitions: {numbers.getNumPartitions()}")

# 2. Create RDD with explicit partitions
# YOUR CODE: Create the same list with exactly 4 partitions
partitions = sc.parallelize([1, 2, 3, 4, 5, 6, 7, 8, 9, 10], 4)


# 3. Create RDD from a range
# YOUR CODE: Create RDD from range(1, 101)
range = sc.parallelize(range(1, 101))

# ------ part 2 -------
# Given: numbers RDD [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

# Task A: Square each number
# Expected: [1, 4, 9, 16, 25, 36, 49, 64, 81, 100]
squared = numbers.map(lambda x: x**2)
print(f"squared: {squared.collect()}")

# Task B: Convert to strings with prefix
# Expected: ["num_1", "num_2", "num_3", ...]
prefixed = numbers.map(lambda x: f"num_{x}")


#-------------- part 3 ----------------
# Task A: Keep only even numbers
# Expected: [2, 4, 6, 8, 10]
evens = numbers.filter(lambda x: x%2 == 0) # YOUR CODE
print(f"evens: {evens.collect()}")

# Task B: Keep numbers greater than 5
# Expected: [6, 7, 8, 9, 10]
greater_than_5 = numbers.filter(lambda x: x>5)# YOUR CODE
print(f">5: {greater_than_5.collect()}")

# Task C: Combine - even AND greater than 5
# Expected: [6, 8, 10]
combined = numbers.filter(lambda x: x>5).filter(lambda x: x%2 == 0)
print(f"combined: {combined.collect()}")



# ----------- part 4 ------------
# Given sentences
sentences = sc.parallelize([
    "Hello World",
    "Apache Spark is Fast",
    "PySpark is Python plus Spark"
])

# Task A: Split into words (use flatMap)
# Expected: ["Hello", "World", "Apache", "Spark", ...]
words = sentences.flatMap(lambda line: line.split())# YOUR CODE
print(f"split: {words.collect()}")

# Task B: Create pairs of (word, length)
# Expected: [("Hello", 5), ("World", 5), ...]
word_lengths = sentences.flatMap(lambda line: [(word, len(word)) for word in line.split()])# YOUR CODE
print(f"pairs: {word_lengths.collect()}")



# --------- part 5 -----------
# Given: log entries
logs = sc.parallelize([
    "INFO: User logged in",
    "ERROR: Connection failed",
    "INFO: Data processed",
    "ERROR: Timeout occurred",
    "DEBUG: Cache hit"
])

# Pipeline: Extract only ERROR messages, convert to uppercase words
# 1. Filter to keep only ERROR lines
# 2. Split each line into words
# 3. Convert each word to uppercase
# Expected: ["ERROR:", "CONNECTION", "FAILED", "ERROR:", "TIMEOUT", "OCCURRED"]
error_words = (logs
               .filter(lambda line: line.startswith("ERROR:"))
               .flatMap(lambda x: x.split())
               .map(lambda word: word.upper()))
print(f"error: {error_words.collect()}")
# YOUR PIPELINE
sc.stop()

Numbers: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
Partitions: 20


squared: [1, 4, 9, 16, 25, 36, 49, 64, 81, 100]
evens: [2, 4, 6, 8, 10]
>5: [6, 7, 8, 9, 10]
combined: [6, 8, 10]
split: ['Hello', 'World', 'Apache', 'Spark', 'is', 'Fast', 'PySpark', 'is', 'Python', 'plus', 'Spark']
pairs: [('Hello', 5), ('World', 5), ('Apache', 6), ('Spark', 5), ('is', 2), ('Fast', 4), ('PySpark', 7), ('is', 2), ('Python', 6), ('plus', 4), ('Spark', 5)]
error: ['ERROR:', 'CONNECTION', 'FAILED', 'ERROR:', 'TIMEOUT', 'OCCURRED']


In [3]:
# transoformations

from pyspark import SparkContext

sc = SparkContext("local[*]", "Transformations")

# Sample log data
logs = sc.parallelize([
    "2024-01-15 10:00:00 INFO User login: alice",
    "2024-01-15 10:01:00 ERROR Database connection failed",
    "2024-01-15 10:02:00 INFO User login: bob",
    "2024-01-15 10:03:00 WARN Memory usage high",
    "2024-01-15 10:04:00 ERROR Timeout occurred",
    "2024-01-15 10:05:00 INFO Data processed: 1000 records",
    "2024-01-15 10:06:00 DEBUG Cache hit rate: 95%"
])

# Task A: Filter only ERROR logs
errors = logs.filter(lambda line: line.split()[2] == "ERROR")# YOUR CODE
print(f"Errors: {errors.collect()}")

# Task B: Extract just the log level from each line
# Expected: ["INFO", "ERROR", "INFO", "WARN", "ERROR", "INFO", "DEBUG"]
levels = logs.map(lambda line: line.split()[2])# YOUR CODE (hint: split and take index 2)
print(f"Levels: {levels.collect()}")

# Task C: Chain - get messages from ERROR logs only
# Expected: ["Database connection failed", "Timeout occurred"]
error_messages = (logs.filter(lambda line: line.split()[2] == "ERROR")
                  .map(lambda line: " ".join(line.split()[3:])))# YOUR CODE
print(f"Error messages: {error_messages.collect()}")


# ----- task 2 ------
# Sample word data
words = sc.parallelize([
    "spark", "hadoop", "spark", "data", "hadoop", 
    "spark", "python", "data", "spark", "scala"
])

# Task A: distinct() - Get unique words
unique = words.distinct()# YOUR CODE
print(f"Unique words: {unique.collect()}")

# Task B: Group and count (wide transformation)
word_counts = words.map(lambda w: (w, 1)).reduceByKey(lambda a, b: a + b)
print(f"Word counts: {word_counts.collect()}")

# Task C: sortBy - Sort by count descending
sorted_counts = word_counts.map(lambda x: (x[1], x[0])).sortByKey(ascending = False) # YOUR CODE (sort by count, descending)
print(f"Sorted: {sorted_counts.collect()}")



# ------------- task 3 ------------------
# Two datasets
set1 = sc.parallelize([1, 2, 3, 4, 5])
set2 = sc.parallelize([4, 5, 6, 7, 8])

# Task A: union() - Combine both sets
combined = set1.union(set2)# YOUR CODE
print(f"Union: {combined.collect()}")

# Task B: intersection() - Common elements
common = set1.intersection(set2)# YOUR CODE
print(f"Intersection: {common.collect()}")

# Task C: subtract() - Elements in set1 but not in set2
difference = set1.subtract(set2)# YOUR CODE
print(f"Difference: {difference.collect()}")





## ------------- task 4 ------------------
# Given: Web server logs
web_logs = sc.parallelize([
    "192.168.1.1 GET /home 200 150ms",
    "192.168.1.2 GET /products 200 230ms",
    "192.168.1.1 POST /login 200 180ms",
    "192.168.1.3 GET /home 404 50ms",
    "192.168.1.2 GET /products 200 210ms",
    "192.168.1.1 GET /home 200 120ms",
    "192.168.1.4 GET /admin 403 30ms"
])

# Build a pipeline to:
# 1. Filter only successful requests (status 200)
# 2. Extract the endpoint (e.g., /home)
# 3. Count requests per endpoint
# 4. Sort by count descending

# YOUR PIPELINE CODE HERE
# Expected output: [('/products', 2), ('/home', 2), ('/login', 1)]
pipeline = web_logs.filter(lambda line: line.split()[3] == "200")

extract = pipeline.map(lambda line: (line.split()[2], 1)).reduceByKey(lambda a, b: a + b)
final = extract.map(lambda x: (x[1], x[0])).sortByKey(ascending = False)
finalv2 = final.map(lambda x: (x[1], x[0])) # flipping the tuple back around
print(f"pipeline: {finalv2.collect()}")
sc.stop()

Errors: ['2024-01-15 10:01:00 ERROR Database connection failed', '2024-01-15 10:04:00 ERROR Timeout occurred']
Levels: ['INFO', 'ERROR', 'INFO', 'WARN', 'ERROR', 'INFO', 'DEBUG']
Error messages: ['Database connection failed', 'Timeout occurred']
Unique words: ['spark', 'hadoop', 'scala', 'python', 'data']
Word counts: [('spark', 4), ('hadoop', 2), ('scala', 1), ('python', 1), ('data', 2)]
Sorted: [(4, 'spark'), (2, 'hadoop'), (2, 'data'), (1, 'scala'), (1, 'python')]
Union: [1, 2, 3, 4, 5, 4, 5, 6, 7, 8]


Intersection: [4, 5]


Difference: [1, 2, 3]
pipeline: [('/products', 2), ('/home', 2), ('/login', 1)]
